# 🛠️ Adım 2: Özellik Mühendisliği (Feature Engineering)
Bu notebook, makine öğrenmesi modelleri için 10 adet yeni özellik üretim sürecini içerir.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import Window

spark = SparkSession.builder.appName("FeatureEng").getOrCreate()

## 1. Veri Hazırlama

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/app/dataset/bankdataset.csv")
df = df.withColumnRenamed("Value", "value").withColumnRenamed("Transaction_count", "transaction_count")

## 2. Yeni Özelliklerin Üretimi
- İşlem başına değer
- Tarihsel özellikler (Ay, Gün, Çeyrek)
- Haftasonu kontrolü

In [ ]:
df = df.withColumn("value_per_transaction", round(col("value") / col("transaction_count"), 2))
df = df.withColumn("parsed_date", to_date(col("Date"), "yyyy-MM-dd"))
df = df.withColumn("month", month("parsed_date")).withColumn("day_of_week", dayofweek("parsed_date"))
df.select("value", "transaction_count", "value_per_transaction", "month", "day_of_week").show(5)

## 3. İstatistiksel Özellikler
Lokasyon bazlı ortalama işlem değerleri (Window Function).

In [ ]:
loc_window = Window.partitionBy("Location")
df = df.withColumn("location_avg_value", round(avg("value").over(loc_window), 2))
df.select("Location", "value", "location_avg_value").show(5)